# SpecDist — Kaggle Quickstart

**Free GPU: 30 h/week · T4 · 12 h sessions** (vs ~90 min idle timeout on free Colab)

| Cell | What it does | Time |
|------|-------------|------|
| 0. Bootstrap | One-shot: clone → deps → auth → run | ~3 min to start |
| 1. Resume | After session restart — pipeline skips completed steps | ~1 min + rest |
| 2. Monitor | State + log tail (auto-refresh option) | instant |
| 3. Save | Verify artifacts before session ends | instant |

---

### Before you start
1. **GPU**: Settings (top right) → Accelerator → **GPU T4 × 1**
2. **Internet**: Settings → Internet → **On**
3. **Kaggle Secrets** (Add-ons → Secrets in the left sidebar):
   - `WANDB_API_KEY` — https://wandb.ai/authorize
   - `HF_TOKEN` — https://huggingface.co/settings/tokens  _(public Qwen3 models work without it)_
   - `GITHUB_TOKEN` — required only if the repo is private
4. **Run all**: Shift+F5 (or Run → Run All)

### Storage
- `/kaggle/working/` — 20 GB scratch per session (wiped on restart)
- To persist checkpoints: after the run completes, go to **Notebook → Data → Output → + New Dataset**
  to save `/kaggle/working/` as a Kaggle dataset. Re-attach it next session — Cell 1 restores it automatically.

### Tip: pre-upload model weights (skip the 5-10 min download)
1. On any machine: `huggingface-cli download Qwen/Qwen3-0.6B Qwen/Qwen3-4B`
2. Upload `~/.cache/huggingface/` as a private Kaggle dataset (100 GB quota, 20 GB/dataset)
3. Attach the dataset to this notebook, set `KAGGLE_HF_DATASET` in Cell 0 to the mount path

### CONFIG options
| CONFIG | Teacher | Steps | Est. time on T4 | Purpose |
|--------|---------|-------|-----------------|--------|
| `colab_lite` | Qwen3-1.7B (BF16) | 300 | ~25 min | Quick trend check |
| `colab` | Qwen3-4B (4-bit NF4) | 500 | ~4 h | Production T4 results |

In [ ]:
# =============================================================================
# Cell 0 — BOOTSTRAP  (run once per fresh session)
# Edit the variables below, then Shift+F5 (Run All).
# After a session restart, use Cell 1 (Resume) instead — it is self-contained.
# =============================================================================

# -- Edit these ---------------------------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"

# colab_lite: 1.7B teacher, 300 steps, ~25 min  (quick trend check)
# colab:      4B  teacher, 4-bit NF4, 500 steps, ~4 h  (production results)
CONFIG      = "colab"
SMOKE       = False    # True = 10-step crash check (~5 min); run this first!
BACKGROUND  = True     # True = pipeline runs in background; monitor with Cell 2
LOSSES      = None     # None = all losses  |  "kl,ebe" = subset
EXTRA_ARGS  = []

# Optional: path to a pre-attached Kaggle dataset containing HF model weights.
# Set to the dataset mount path to skip the ~5-10 min model download.
# Example: "/kaggle/input/qwen3-hf-cache"
# Leave as None to download models to /kaggle/working/specdist/hf_cache/.
KAGGLE_HF_DATASET = None
# -----------------------------------------------------------------------------

import os, subprocess, sys

# -- Keep-alive: prevent Kaggle idle-timeout (45 s JS heartbeat) --------------
try:
    from IPython.display import display, Javascript
    display(Javascript("""
(function(){
  if(window.__sd_ka)return;
  window.__sd_ka=setInterval(function(){
    document.dispatchEvent(new MouseEvent('mousemove',{bubbles:true}));
  },45000);
  console.log('[specdist] keep-alive on (45 s)');
})();
"""))
    print("[keep-alive] JS heartbeat started")
except Exception:
    pass  # not in a browser — silently skip

# -- Kaggle secrets → env  (must run before any colab_utils import) -----------
def _kread(name):
    """Read a Kaggle secret, fall back to env var."""
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

for _k in ("WANDB_API_KEY", "HF_TOKEN", "GITHUB_TOKEN"):
    _v = _kread(_k)
    if _v:
        os.environ[_k] = _v
        print(f"  {_k}: set")
    else:
        print(f"  {_k}: not found (Add-ons -> Secrets)")

# -- Clone / update repo -------------------------------------------------------
gh = os.environ.get("GITHUB_TOKEN", "")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh:  # keep token current in the remote URL in case it rotated
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    pull_r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                             capture_output=True, text=True)
    print(pull_r.stdout.strip() or "Repo already up to date")
    if pull_r.returncode != 0:
        print("[pull error]", pull_r.stderr.strip())

# Flush stale module cache so updated colab_utils is loaded fresh after pull.
# Without this, Python serves the old in-memory module even if the file changed.
sys.modules.pop("colab_utils", None)

# -- Dirs ----------------------------------------------------------------------
os.makedirs(f"{STORAGE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{STORAGE_ROOT}/logs", exist_ok=True)
os.chdir(GBV_DIR)
sys.path.insert(0, f"{GBV_DIR}/deploy")

# -- Import shared utils (repo is now current) ---------------------------------
from colab_utils import (install_deps, fetch_training_data, setup_hf_cache,
                          prefetch_models, auth_wandb, auth_hf, check_gpu,
                          run_pipeline)

# -- Install deps --------------------------------------------------------------
install_deps(GBV_DIR)

# -- HF model cache ------------------------------------------------------------
if KAGGLE_HF_DATASET and os.path.isdir(KAGGLE_HF_DATASET):
    # Use pre-uploaded model weights dataset — zero download time.
    # Requires the dataset to be structured as a standard HF Hub cache.
    os.environ["HF_HOME"]            = KAGGLE_HF_DATASET
    os.environ["TRANSFORMERS_CACHE"] = KAGGLE_HF_DATASET
    for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
        os.environ.pop(_flag, None)
    print(f"[cache] Using attached dataset: {KAGGLE_HF_DATASET}")
else:
    # /kaggle/working/specdist/hf_cache/ — up to ~20 GB (ephemeral per session).
    # Small models (<=5 GB) download in ~1-2 min; 4B takes ~5-10 min first run.
    setup_hf_cache(STORAGE_ROOT)
    prefetch_models(CONFIG, GBV_DIR, STORAGE_ROOT)

# -- Auth + data ---------------------------------------------------------------
auth_wandb()
auth_hf()
fetch_training_data(GBV_DIR)   # downloads gsm8k_train.jsonl if missing
check_gpu(warn_below_gb=12.0)
print()

# -- Run pipeline --------------------------------------------------------------
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR,
                 smoke=SMOKE, losses=LOSSES,
                 background=BACKGROUND, extra_args=EXTRA_ARGS)

In [ ]:
# =============================================================================
# Cell 1 — RESUME  (after session restart or interrupted pipeline)
# Self-contained: works correctly even if Cell 0 never ran this session.
# The pipeline reads the state file and skips already-completed steps.
# =============================================================================

# -- Edit these (must match Cell 0) -------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"
CONFIG       = "colab"   # must match the CONFIG used in Cell 0
KAGGLE_HF_DATASET = None  # same as Cell 0
# -----------------------------------------------------------------------------

import os, subprocess, sys, shutil

# -- Keep-alive ----------------------------------------------------------------
try:
    from IPython.display import display, Javascript
    display(Javascript("""
(function(){
  if(window.__sd_ka)return;
  window.__sd_ka=setInterval(function(){
    document.dispatchEvent(new MouseEvent('mousemove',{bubbles:true}));
  },45000);
  console.log('[specdist] keep-alive on (45 s)');
})();
"""))
    print("[keep-alive] JS heartbeat started")
except Exception:
    pass

# -- Secrets → env -------------------------------------------------------------
def _kread(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

for _k in ("WANDB_API_KEY", "HF_TOKEN", "GITHUB_TOKEN"):
    _v = _kread(_k)
    if _v:
        os.environ[_k] = _v

# -- Clone / update repo -------------------------------------------------------
gh = os.environ.get("GITHUB_TOKEN", "")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    pull_r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                             capture_output=True, text=True)
    print(pull_r.stdout.strip() or "Repo already up to date")
    if pull_r.returncode != 0:
        print("[pull error]", pull_r.stderr.strip())

# Flush stale module cache — without this, Python serves the old in-memory
# module even if the file on disk changed after the pull.
sys.modules.pop("colab_utils", None)

# -- Reinstall deps (always wiped on session restart) --------------------------
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", f"{REPO_DIR}/gbv-research/requirements.txt",
     "bitsandbytes", "accelerate", "torchao>=0.16.0"],
    check=True,
)
print("Deps installed")

# -- Restore checkpoints from an attached input dataset (if any) ---------------
# If you saved a previous session's output as a Kaggle dataset and re-attached
# it (Add Data -> Your Datasets), this copies the checkpoints into STORAGE_ROOT
# so the pipeline can resume.  Tries a few common dataset name variants.
os.makedirs(f"{STORAGE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{STORAGE_ROOT}/logs", exist_ok=True)

_restored = False
for _ds_name in ("specdist-checkpoints", "specdist_checkpoints",
                 "specdist-output",      "specdist"):
    _ds_path = f"/kaggle/input/{_ds_name}"
    if os.path.isdir(_ds_path):
        _ckpt_dst = f"{STORAGE_ROOT}/checkpoints"
        _db_src   = os.path.join(_ds_path, "specdist", "results.db")
        _db_dst   = os.path.join(STORAGE_ROOT, "results.db")
        # Restore results.db if not already present
        if os.path.exists(_db_src) and not os.path.exists(_db_dst):
            shutil.copy2(_db_src, _db_dst)
            print(f"Restored results.db from {_db_src}")
        # Restore checkpoint directories
        for _item in os.listdir(_ds_path):
            _src = os.path.join(_ds_path, _item)
            _dst = os.path.join(_ckpt_dst, _item)
            if not os.path.exists(_dst):
                if os.path.isdir(_src):
                    shutil.copytree(_src, _dst)
                else:
                    shutil.copy2(_src, _dst)
        _items = os.listdir(_ckpt_dst)
        print(f"Checkpoints restored from {_ds_path}: {_items}")
        _restored = True
        break

if not _restored:
    print("No checkpoint dataset attached — starting from scratch")
    print("  (To attach: Notebook -> Data -> + Add Data -> Your Datasets)")

# -- Paths + sys.path ----------------------------------------------------------
os.chdir(GBV_DIR)
sys.path.insert(0, f"{GBV_DIR}/deploy")

# -- Import shared utils (safe — repo is current and module cache flushed) -----
from colab_utils import (auth_wandb, auth_hf, setup_hf_cache, prefetch_models,
                          fetch_training_data, run_pipeline)

# -- HF model cache ------------------------------------------------------------
if KAGGLE_HF_DATASET and os.path.isdir(KAGGLE_HF_DATASET):
    os.environ["HF_HOME"]            = KAGGLE_HF_DATASET
    os.environ["TRANSFORMERS_CACHE"] = KAGGLE_HF_DATASET
    for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
        os.environ.pop(_flag, None)
    print(f"[cache] Using attached dataset: {KAGGLE_HF_DATASET}")
else:
    setup_hf_cache(STORAGE_ROOT)
    prefetch_models(CONFIG, GBV_DIR, STORAGE_ROOT)

# -- Auth + data ---------------------------------------------------------------
auth_wandb()
auth_hf()
fetch_training_data(GBV_DIR)   # re-downloads gsm8k_train.jsonl if wiped

# -- Resume pipeline (background so Cell 2 monitor can run in parallel) --------
print()
print(f"Resuming config={CONFIG} — completed steps are skipped automatically.")
print()
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR, background=True)

In [ ]:
# =============================================================================
# Cell 2 — MONITOR  (safe to run any time, including while pipeline runs)
# Set AUTO_REFRESH = True for a live tail; interrupt the cell to stop.
# CONFIG must match the config used in Cell 0 or Cell 1.
# =============================================================================
import sys

REPO_DIR     = "/kaggle/working/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = "/kaggle/working/specdist"
CONFIG       = "colab"   # must match Cell 0/1
AUTO_REFRESH = False     # True = live tail loop (interrupt cell to stop)
REFRESH_SECS = 20

sys.path.insert(0, f"{GBV_DIR}/deploy")
from colab_utils import monitor

monitor(STORAGE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)

In [ ]:
# =============================================================================
# Cell 3 — SAVE  (verify artifacts before the session ends)
#
# Kaggle saves /kaggle/working/ automatically when the notebook session ends.
# To persist checkpoints across sessions:
#   1. Notebook → Data → Output → + New Dataset
#   2. Name it "specdist-checkpoints"
#   3. In future sessions: Add Data → Your Datasets → attach it
#      Cell 1 (Resume) will restore the checkpoints automatically.
# =============================================================================
import os, json, pathlib

STORAGE_ROOT = "/kaggle/working/specdist"

ckpt_dir = os.path.join(STORAGE_ROOT, "checkpoints")
db_path  = os.path.join(STORAGE_ROOT, "results.db")
log_path = os.path.join(STORAGE_ROOT, "logs", "pipeline_output.log")

status = {
    "checkpoints": os.listdir(ckpt_dir) if os.path.isdir(ckpt_dir) else [],
    "results_db_bytes": os.path.getsize(db_path) if os.path.exists(db_path) else 0,
    "log_lines": sum(1 for _ in open(log_path)) if os.path.exists(log_path) else 0,
}
status_path = os.path.join(STORAGE_ROOT, "run_status.json")
with open(status_path, "w") as f:
    json.dump(status, f, indent=2)

print(f"Artifacts at: {STORAGE_ROOT}")
print(f"  results.db   : {status['results_db_bytes']:,} bytes")
print(f"  checkpoints/ : {status['checkpoints']}")
print(f"  log lines    : {status['log_lines']}")
print()

# Total /kaggle/working/ usage
try:
    total = sum(
        f.stat().st_size
        for f in pathlib.Path("/kaggle/working").rglob("*")
        if f.is_file()
    )
    print(f"Total /kaggle/working/ usage: {total / 1024**3:.2f} GB (limit 20 GB)")
except Exception as e:
    print(f"Could not compute disk usage: {e}")

print()
print("To persist across sessions:")
print("  Notebook -> Data -> Output -> + New Dataset (saves /kaggle/working/)")
print("  Name it 'specdist-checkpoints'.")
print("  Attach in the next session -> Cell 1 restores checkpoints automatically.")